# Naive Bayes — Bayes' rule with a (very) convenient assumption

> Tutorial pair for [`naive_bayes.py`](naive_bayes.py).

## 1. Intuition
We want $P(\text{class}\mid\text{features})$. Bayes' rule flips it into something
we *can* estimate: $P(\text{features}\mid\text{class})\,P(\text{class})$. The hard
part is the joint $P(x_1,\dots,x_d\mid c)$. Naive Bayes makes a sweeping
assumption — given the class, features are **independent** — so that joint
collapses into a product of one-dimensional pieces we can count directly. The
assumption is almost always false, yet the *argmax* it produces is often right,
which is why NB is a famously strong, dirt-cheap baseline (especially for text).

## 2. Concept (the slide)
- **Generative model:** each class has a story for generating features;
  classify by asking which class most likely generated $x$.
- **Conditional independence:** $P(x\mid c)=\prod_j P(x_j\mid c)$ — turns a
  $d$-dimensional density into $d$ tiny ones.
- **Event models (the three flavours):**
  - *Gaussian* — continuous $x_j$: $P(x_j\mid c)=\mathcal N(\mu_{cj},\sigma^2_{cj})$.
  - *Multinomial* — counts (bag-of-words): $P(x\mid c)\propto\prod_j P(j\mid c)^{x_j}$.
  - *Bernoulli* — binary present/absent, with an explicit *absence* term.
- **Fit = count.** Parameters are closed-form MLEs; no gradient descent.
- **Work in log-space** and **smooth** to dodge underflow and zero-probabilities.

## 3. Math derivation — Bayes' rule, the naive factorization, MLE, smoothing

### Posterior via Bayes
$$P(y=c\mid x)=\frac{P(y=c)\,P(x\mid y=c)}{P(x)}
 =\frac{P(y=c)\,P(x\mid y=c)}{\sum_{c'}P(y=c')\,P(x\mid y=c')}.$$
The denominator $P(x)$ is constant across $c$, so for classification
$$\hat y=\arg\max_c\;P(y=c)\,P(x\mid y=c).$$

### The naive (conditional independence) assumption
$$P(x\mid y=c)=\prod_{j=1}^d P(x_j\mid y=c)
\;\Rightarrow\;
\boxed{\;\log P(y=c\mid x)=\log P(y=c)+\sum_{j=1}^d\log P(x_j\mid y=c)-\log Z\;}$$
We compute everything in **log-space** (sums, not products) and recover the
normalizer with log-sum-exp: $\log Z=\log\sum_c \exp(\text{joint}_c)$.

### Maximum-likelihood parameters (per event model)
**Gaussian.** For class $c$, feature $j$:
$$\mu_{cj}=\frac1{n_c}\sum_{i:y_i=c}x_{ij},\qquad
\sigma^2_{cj}=\frac1{n_c}\sum_{i:y_i=c}(x_{ij}-\mu_{cj})^2,$$
$$\log P(x_j\mid c)=-\tfrac12\Big(\log(2\pi\sigma^2_{cj})+\frac{(x_j-\mu_{cj})^2}{\sigma^2_{cj}}\Big).$$

**Multinomial.** $\hat\theta_{cj}=P(j\mid c)$ maximizes $\sum_j N_{cj}\log\theta_{cj}$
under $\sum_j\theta_{cj}=1$. A Lagrange multiplier gives $\theta_{cj}=N_{cj}/N_c$;
with **Laplace ($+\alpha$) smoothing**:
$$\hat\theta_{cj}=\frac{N_{cj}+\alpha}{N_c+\alpha\,d},\qquad
\log P(x\mid c)=\sum_j x_j\log\hat\theta_{cj}.$$

**Bernoulli.** $p_{cj}=P(x_j=1\mid c)$, smoothed:
$$p_{cj}=\frac{(\#\,c\text{-docs with }j)+\alpha}{n_c+2\alpha},\qquad
\log P(x\mid c)=\sum_j\big[x_j\log p_{cj}+(1-x_j)\log(1-p_{cj})\big].$$
The $(1-x_j)\log(1-p_{cj})$ term — *crediting absent words* — is exactly what
distinguishes Bernoulli from Multinomial.

### Why smoothing is mandatory
Without it, a single feature unseen in class $c$ gives $P(j\mid c)=0$, which
makes the *entire* product zero ($\log\to-\infty$) and vetoes class $c$ no matter
what the other features say. Adding $\alpha$ pseudo-counts (a Dirichlet/Beta prior
→ MAP estimate) keeps every probability strictly positive.

## 4. NumPy implementation — Gaussian / Multinomial / Bernoulli from scratch

In [ ]:
# ===== actual implementation from naive_bayes.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def _logsumexp(A, axis=1):
    """Numerically stable log(sum(exp(A))) along `axis`."""
    m = A.max(axis=axis, keepdims=True)
    return (m.squeeze(axis) + np.log(np.exp(A - m).sum(axis=axis)))

class GaussianNB:
    r"""
    Continuous features. Per class c and feature j, model x_j ~ N(mu_cj, var_cj).

    MLE on the training points of class c:
        mu_cj  = mean_i x_ij,   var_cj = mean_i (x_ij - mu_cj)^2.
    Log-likelihood of a feature:
        log N(x; mu, var) = -1/2 [ log(2π var) + (x-mu)^2 / var ].
    """

    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing      # add to variances for stability

    def fit(self, X, y):
        X = np.asarray(X, float)
        self.classes_ = np.unique(y)
        n = len(X)
        self.theta_ = []     # mean per class:  (n_classes, n_features)
        self.sigma_ = []     # variance per class
        self.log_prior_ = []
        eps = self.var_smoothing * X.var(0).max()
        for c in self.classes_:
            Xc = X[y == c]
            self.theta_.append(Xc.mean(0))
            self.sigma_.append(Xc.var(0) + eps)
            self.log_prior_.append(np.log(len(Xc) / n))     # log P(y=c)
        self.theta_ = np.array(self.theta_)
        self.sigma_ = np.array(self.sigma_)
        self.log_prior_ = np.array(self.log_prior_)
        return self

    def _joint_log_likelihood(self, X):
        # log P(y=c) + Σ_j log N(x_j; mu_cj, var_cj), for every class c
        jll = []
        for k in range(len(self.classes_)):
            mu, var = self.theta_[k], self.sigma_[k]
            ll = -0.5 * (np.log(2 * np.pi * var) + (X - mu) ** 2 / var).sum(1)
            jll.append(self.log_prior_[k] + ll)
        return np.array(jll).T          # (n_samples, n_classes)

    def predict_log_proba(self, X):
        jll = self._joint_log_likelihood(np.asarray(X, float))
        return jll - _logsumexp(jll, axis=1)[:, None]      # normalize -> log P(y|x)

    def predict(self, X):
        return self.classes_[self._joint_log_likelihood(np.asarray(X, float)).argmax(1)]

class MultinomialNB:
    r"""
    Count features (e.g. bag-of-words term frequencies). Each class is a
    multinomial over the vocabulary.

    With Laplace (additive) smoothing alpha:
        P(j | c) = (count of feature j in class c + alpha)
                   / (total counts in class c + alpha * n_features).
    For a document x (a count vector):
        log P(x | c) = Σ_j x_j * log P(j | c)   (multinomial coeff drops in argmax).
    """

    def __init__(self, alpha=1.0):
        self.alpha = alpha             # smoothing pseudo-count

    def fit(self, X, y):
        X = np.asarray(X, float)
        self.classes_ = np.unique(y)
        n, d = X.shape
        self.feature_log_prob_ = []    # log P(j | c)
        self.log_prior_ = []
        for c in self.classes_:
            Xc = X[y == c]
            counts = Xc.sum(0) + self.alpha          # smoothed per-feature counts
            self.feature_log_prob_.append(np.log(counts) - np.log(counts.sum()))
            self.log_prior_.append(np.log(len(Xc) / n))
        self.feature_log_prob_ = np.array(self.feature_log_prob_)
        self.log_prior_ = np.array(self.log_prior_)
        return self

    def _joint_log_likelihood(self, X):
        # log P(y=c) + Σ_j x_j log P(j|c)  ==  prior + X @ feature_log_prob^T
        return X @ self.feature_log_prob_.T + self.log_prior_

    def predict_log_proba(self, X):
        jll = self._joint_log_likelihood(np.asarray(X, float))
        return jll - _logsumexp(jll, axis=1)[:, None]

    def predict(self, X):
        return self.classes_[self._joint_log_likelihood(np.asarray(X, float)).argmax(1)]

class BernoulliNB:
    r"""
    Binary features (presence/absence). Each feature is a Bernoulli per class.

    Smoothed parameter:
        p_cj = (count of docs in class c with feature j + alpha)
               / (n docs in class c + 2 alpha).
    Likelihood (note the explicit ABSENT term — what makes it differ from
    Multinomial):
        log P(x | c) = Σ_j [ x_j log p_cj + (1 - x_j) log(1 - p_cj) ].
    """

    def __init__(self, alpha=1.0, binarize=0.0):
        self.alpha = alpha
        self.binarize = binarize       # threshold to binarize inputs

    def _binarize(self, X):
        X = np.asarray(X, float)
        return (X > self.binarize).astype(float) if self.binarize is not None else X

    def fit(self, X, y):
        X = self._binarize(X)
        self.classes_ = np.unique(y)
        n = len(X)
        self.feature_log_prob_ = []        # log p_cj
        self.neg_log_prob_ = []            # log(1 - p_cj)
        self.log_prior_ = []
        for c in self.classes_:
            Xc = X[y == c]
            p = (Xc.sum(0) + self.alpha) / (len(Xc) + 2 * self.alpha)
            self.feature_log_prob_.append(np.log(p))
            self.neg_log_prob_.append(np.log(1 - p))
            self.log_prior_.append(np.log(len(Xc) / n))
        self.feature_log_prob_ = np.array(self.feature_log_prob_)
        self.neg_log_prob_ = np.array(self.neg_log_prob_)
        self.log_prior_ = np.array(self.log_prior_)
        return self

    def _joint_log_likelihood(self, X):
        X = self._binarize(X)
        # Σ_j x_j log p + (1-x_j) log(1-p) = X@logp^T + (1-X)@log(1-p)^T
        present = X @ self.feature_log_prob_.T
        absent = (1 - X) @ self.neg_log_prob_.T
        return present + absent + self.log_prior_

    def predict_log_proba(self, X):
        jll = self._joint_log_likelihood(X)
        return jll - _logsumexp(jll, axis=1)[:, None]

    def predict(self, X):
        return self.classes_[self._joint_log_likelihood(X).argmax(1)]

## 5. PyTorch implementation — vectorized log-probabilities on tensors

In [ ]:
# ===== actual implementation from naive_bayes.py =====
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

class GaussianNBTorch:
    r"""
    Gaussian Naive Bayes with the same closed-form MLE, but the parameter fit and
    the per-class log-likelihood are computed as **vectorized tensor ops** and the
    posterior is normalized with `torch.logsumexp`. No autograd / gradients —
    Naive Bayes has a closed-form solution; we just lean on tensor broadcasting.
    """

    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing

    def fit(self, X, y):
        dev = get_device(); self.device = dev
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        y = torch.as_tensor(np.asarray(y), device=dev)
        self.classes_ = torch.unique(y)
        eps = self.var_smoothing * X.var(0).max()
        means, vars, log_priors = [], [], []
        for c in self.classes_:
            Xc = X[y == c]
            means.append(Xc.mean(0))
            vars.append(Xc.var(0, unbiased=False) + eps)
            log_priors.append(torch.log(torch.tensor(len(Xc) / len(X), device=dev)))
        self.theta_ = torch.stack(means)            # (C, d)
        self.sigma_ = torch.stack(vars)             # (C, d)
        self.log_prior_ = torch.stack(log_priors)   # (C,)
        return self

    def _jll(self, X):
        # Broadcast over classes: X (n,1,d) vs params (1,C,d) -> (n, C)
        Xe = X[:, None, :]
        mu = self.theta_[None, :, :]
        var = self.sigma_[None, :, :]
        ll = -0.5 * (torch.log(2 * np.pi * var) + (Xe - mu) ** 2 / var).sum(-1)
        return ll + self.log_prior_[None, :]

    @torch.no_grad()
    def predict_log_proba(self, X):
        X = torch.as_tensor(X, dtype=torch.float32, device=self.device)
        jll = self._jll(X)
        return (jll - torch.logsumexp(jll, dim=1, keepdim=True)).cpu().numpy()

    @torch.no_grad()
    def predict(self, X):
        X = torch.as_tensor(X, dtype=torch.float32, device=self.device)
        idx = self._jll(X).argmax(1)
        return self.classes_[idx].cpu().numpy()

def _toy_text():
    """Tiny bag-of-words corpus: 2 classes (sports vs tech), 6-word vocabulary."""
    vocab = ["ball", "game", "score", "cpu", "code", "data"]
    docs = [
        ("ball game score score", 0), ("game ball game", 0),
        ("score ball game ball", 0), ("game game score", 0),
        ("cpu code data data", 1), ("code cpu code", 1),
        ("data data code cpu", 1), ("cpu cpu code", 1),
    ]
    idx = {w: i for i, w in enumerate(vocab)}
    X = np.zeros((len(docs), len(vocab)))
    y = np.zeros(len(docs), int)
    for r, (text, label) in enumerate(docs):
        for w in text.split():
            X[r, idx[w]] += 1
        y[r] = label
    return X, y, vocab

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)

    # --- Gaussian NB on iris (continuous features) ---
    from sklearn.datasets import load_iris
    iris = load_iris()
    Xi, yi = iris.data, iris.target
    rng = np.random.default_rng(SEED)
    perm = rng.permutation(len(Xi))
    Xi, yi = Xi[perm], yi[perm]
    Xtr, ytr, Xte, yte = Xi[:120], yi[:120], Xi[120:], yi[120:]

    print("=== Gaussian NB on iris (continuous) ===")
    g = GaussianNB().fit(Xtr, ytr)
    print(f"  NumPy GaussianNB  test acc = {np.mean(g.predict(Xte) == yte):.3f}")
    gt = GaussianNBTorch().fit(Xtr, ytr)
    print(f"  Torch GaussianNB  test acc = {np.mean(gt.predict(Xte) == yte):.3f}")
    # the two should agree on log-probabilities
    diff = np.abs(g.predict_log_proba(Xte) - gt.predict_log_proba(Xte)).max()
    print(f"  max |logP_numpy - logP_torch| = {diff:.2e}  (same model)")

    # --- Multinomial & Bernoulli NB on a tiny bag-of-words corpus ---
    X, y, vocab = _toy_text()
    print("\n=== Text NB on tiny bag-of-words (sports=0 vs tech=1) ===")
    mnb = MultinomialNB(alpha=1.0).fit(X, y)
    bnb = BernoulliNB(alpha=1.0).fit(X, y)
    print(f"  MultinomialNB train acc = {np.mean(mnb.predict(X) == y):.3f}")
    print(f"  BernoulliNB   train acc = {np.mean(bnb.predict(X) == y):.3f}")

    tests = ["game ball score", "cpu code data", "game code"]
    idx = {w: i for i, w in enumerate(vocab)}
    Xt = np.zeros((len(tests), len(vocab)))
    for r, t in enumerate(tests):
        for w in t.split():
            Xt[r, idx[w]] += 1
    names = {0: "sports", 1: "tech"}
    for t, p, lp in zip(tests, mnb.predict(Xt), mnb.predict_log_proba(Xt)):
        print(f"    '{t:18s}' -> {names[p]:6s}  P={np.exp(lp.max()):.3f}")

    # --- show Laplace smoothing prevents zero probabilities ---
    print("\n  Laplace smoothing: P(word|class) for class 'tech' (alpha=1):")
    for w, lp in zip(vocab, mnb.feature_log_prob_[1]):
        print(f"    {w:6s}: {np.exp(lp):.3f}")
    print("  -> 'ball'/'game' never appear in tech docs yet get nonzero prob.")

## 6. Train — Gaussian NB on iris; Multinomial/Bernoulli on bag-of-words

In [ ]:
demo()

## 7. Visualization — Gaussian NB decision regions & smoothing effect

Left: Gaussian NB on two iris features — note the smooth, quadratic class
boundaries (each class is an axis-aligned Gaussian). Right: per-word
$P(\text{word}\mid\text{class})$ for the toy corpus, showing Laplace smoothing
keeping unseen words nonzero.

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_iris
import naive_bayes as M

iris = load_iris()
X2, y = iris.data[:, [0, 2]], iris.target          # 2 features for plotting
g = M.GaussianNB().fit(X2, y)

xx, yy = np.meshgrid(np.linspace(X2[:,0].min()-.5, X2[:,0].max()+.5, 300),
                     np.linspace(X2[:,1].min()-.5, X2[:,1].max()+.5, 300))
Z = g.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].contourf(xx, yy, Z, alpha=.3, cmap="viridis")
ax[0].scatter(X2[:,0], X2[:,1], c=y, cmap="viridis", s=18, edgecolors="k", linewidths=.3)
ax[0].set_xlabel(iris.feature_names[0]); ax[0].set_ylabel(iris.feature_names[2])
ax[0].set_title("Gaussian NB decision regions")

X, yt, vocab = M._toy_text()
mnb = M.MultinomialNB(alpha=1.0).fit(X, yt)
probs = np.exp(mnb.feature_log_prob_)              # (2 classes, vocab)
w = np.arange(len(vocab)); width = .38
ax[1].bar(w - width/2, probs[0], width, label="sports")
ax[1].bar(w + width/2, probs[1], width, label="tech")
ax[1].set_xticks(w); ax[1].set_xticklabels(vocab, rotation=30)
ax[1].set_ylabel("P(word | class)"); ax[1].set_title("Multinomial NB (Laplace-smoothed)")
ax[1].legend()
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- **Pick the event model for the data**: continuous → Gaussian; word *counts* →
  Multinomial; binary *occurrence* → Bernoulli. They differ in $P(x_j\mid c)$.
- **Always log-space + log-sum-exp**: products of many small probabilities
  underflow to 0; sums of logs don't.
- **Always smooth** ($\alpha>0$): one unseen feature otherwise zeroes a class.
- Probabilities are typically **poorly calibrated** (the independence assumption
  double-counts correlated features → overconfident); trust the *ranking/argmax*,
  not the exact $P$. Calibrate (Platt/isotonic) if you need real probabilities.
- It is linear in $\log$-space and trains in one pass — an excellent fast,
  low-variance baseline, especially for high-dimensional sparse text.